# Getting Started with Claude Opus 5.5 on Amazon Bedrock

**Anthropic's most capable Opus model for agentic coding, knowledge work, and long-running tasks — and the first of the Claude 5.5 family.**

Claude Opus 5.5 does more with fewer tokens than Claude Opus 5, and new pricing (lower per-token rates, much cheaper cache reads) passes those gains through, so the average cost per task drops. It is also trained to communicate more clearly: as it works it surfaces what it did, what it found, and what it needs from you, which makes long-running sessions easier to follow, review, and trust. This notebook onboards you end to end — how to access it, what changed from Opus 5, and use cases that verify their own results in code.

---

## What you'll learn

- Invoking the model three ways: InvokeModel (bedrock-runtime), Converse (bedrock-runtime), and the Anthropic Messages API (bedrock-runtime and bedrock-mantle)
- Adaptive thinking and effort levels — and why thinking can no longer be disabled or budgeted
- Measuring token efficiency and cost per task, including thinking tokens
- The clearer-communication behavior, and how to prompt for it
- Handling refusals, which are more frequent on this model and differ by API surface
- Capabilities in practice (agentic coding, knowledge work, tool use) with checks that grade themselves
- Migrating from Claude Opus 5

## Key capabilities

| | |
|---|---|
| Model IDs (CRIS, `bedrock-runtime`) | `us.anthropic.claude-opus-5-5`, `global.anthropic.claude-opus-5-5` , `eu.anthropic.claude-opus-5-5`, `jp.anthropic.claude-opus-5-5`, `au.anthropic.claude-opus-5-5`|
| Model ID (`bedrock-mantle`) | `anthropic.claude-opus-5-5` |
| APIs on `bedrock-runtime` | Messages API, Converse, InvokeModel |
| APIs on `bedrock-mantle` | Messages API |
| Reasoning | Adaptive thinking, always on and cannot be disabled; effort `low`/`medium`/`high`/`xhigh`/`max`, default `medium` |
| Thinking budgets | Removed — manual `budget_tokens` is deprecated, effort is the control |
| Safety | First Opus model with Claude Fable 5.1-style classifiers in cyber security, biology, and AI development; refuses more than any earlier Opus |
| Efficiency | Fewer tokens per task than Opus 5; lower per-token price and much cheaper cache reads |
| Input / output modalities | Text, Image in; Text out |

---

## When to use Claude Opus 5.5

Reach for Opus 5.5 where consistency and depth matter most. In software development it is an improvement over Opus 5 for longer-running sessions, largely because of the clearer communication and explainability — that makes a multi-hour job easier to use, review, and trust. For knowledge work it needs fewer corrections than Opus 5 when working with and producing long documents and reports. Because the efficiency gains and the lower prices stack, more ambitious agentic work becomes affordable to run at scale.

For short and routine calls, a smaller and faster Claude model remains the better latency and cost choice: Opus 5.5 always reasons before it answers.

## Access prerequisites

1. An active AWS account with Amazon Bedrock access.
2. AWS CLI installed and configured.
3. Python 3.10+.
4. `pip install boto3 "anthropic[bedrock]" aws_bedrock_token_generator`.
5. IAM permissions `bedrock:InvokeModel` and `bedrock:InvokeModelWithResponseStream`.

## Regions

On `bedrock-runtime` the model is served through cross-Region inference, so use the `us.` (US Geo CRIS) or `global.` (Global CRIS) prefixed model ID rather than the bare one. It is available on `bedrock-runtime` in AWS GovCloud (US) as well.

`bedrock-mantle` is an in-Region endpoint, and Opus 5.5 is available on it in:

| Region | Name |
|---|---|
| `us-east-1` | US East (N. Virginia) |
| `ap-southeast-4` | Asia Pacific (Melbourne) |
| `us-gov-west-1` | AWS GovCloud (US-West) |

It is also available through Claude Platform on AWS in North America. Confirm the current list in the [Amazon Bedrock documentation](https://docs.aws.amazon.com/bedrock/latest/userguide/model-cards-anthropic.html).


## 1. Setup

In [ ]:
%pip install --quiet --upgrade boto3 "anthropic[bedrock]" aws_bedrock_token_generator anthropic

In [ ]:
import boto3, json, re
from botocore.config import Config

REGION = "us-east-1"
CRIS_MODEL_ID = "us.anthropic.claude-opus-5-5"         # InvokeModel / Converse / Messages on bedrock-runtime
MANTLE_MODEL_ID = "anthropic.claude-opus-5-5"          # Messages API on bedrock-mantle
MANTLE_REGION = "us-east-1"                            # or ap-southeast-4 / us-gov-west-1
BASELINE_MODEL_ID = "us.anthropic.claude-opus-5"       # for the cost-per-task comparison in section 4

# Opus 5.5 always thinks, so long responses are normal: keep a generous read timeout.
CFG = Config(read_timeout=1200, retries={"max_attempts": 2})

rt = boto3.client("bedrock-runtime", region_name=REGION, config=CFG)
print("boto3", boto3.__version__, "| region", REGION, "| model", CRIS_MODEL_ID)


## 2. Invoking the model

Three API paths reach Opus 5.5. `bedrock-runtime` serves all three — Messages, Converse, and InvokeModel; `bedrock-mantle` serves the Messages API.

Because thinking is always on, a response may include a reasoning block before the text block. **Always select the block whose type is `text`** rather than indexing a fixed position, and remember that `max_tokens` bounds thinking *and* response text together — leave headroom for both.

In [ ]:
PROMPT = "In two sentences, what is Amazon Bedrock?"

# --- InvokeModel (native Anthropic Messages shape) ---
resp = rt.invoke_model(
    modelId=CRIS_MODEL_ID, contentType="application/json", accept="application/json",
    body=json.dumps({"anthropic_version": "bedrock-2023-05-31", "max_tokens": 2048,
                     "messages": [{"role": "user", "content": PROMPT}]}))
result = json.loads(resp["body"].read())
print("InvokeModel:", next(b["text"] for b in result["content"] if b["type"] == "text"))

# --- Converse (unified, multi-model shape) ---
resp = rt.converse(modelId=CRIS_MODEL_ID, messages=[{"role": "user", "content": [{"text": PROMPT}]}],
                   inferenceConfig={"maxTokens": 2048})
blocks = resp["output"]["message"]["content"]
print("Converse:", "\n".join(b.get("text", "") for b in blocks if "text" in b))

In [ ]:
# --- Anthropic Messages API ---
# Existing Anthropic SDK code runs against Bedrock by pointing base_url at the endpoint's
# /anthropic path and authenticating with a short-lived Bedrock token.
from anthropic import Anthropic
from aws_bedrock_token_generator import provide_token

# On bedrock-runtime (CRIS model ID)
rt_messages = Anthropic(base_url=f"https://bedrock-runtime.{REGION}.amazonaws.com/anthropic",
                        api_key=provide_token(region=REGION))
msg = rt_messages.messages.create(model=CRIS_MODEL_ID, max_tokens=2048,
                                  messages=[{"role": "user", "content": PROMPT}])
print("Messages (runtime):", next((b.text for b in msg.content if b.type == "text"), ""))

# On bedrock-mantle (bare model ID, in-Region endpoint at bedrock-mantle.{region}.api.aws)
try:
    from anthropic import AnthropicBedrockMantle
    mantle = AnthropicBedrockMantle(aws_region=MANTLE_REGION)
    msg = mantle.messages.create(model=MANTLE_MODEL_ID, max_tokens=2048,
                                 messages=[{"role": "user", "content": PROMPT}])
    print(f"Messages (mantle, {MANTLE_REGION}):", next((b.text for b in msg.content if b.type == "text"), ""))
except Exception as e:
    # The Mantle catalog can lag the runtime catalog for a new model; the runtime paths above still work.
    print("Mantle path unavailable on this account/Region:", type(e).__name__, str(e)[:140])


## 3. Adaptive thinking and effort

Opus 5.5 is adaptive-thinking-only. It always thinks and decides how much thinking each task needs; **requests cannot disable thinking or set a thinking budget**. Extended thinking with a manual `budget_tokens` is deprecated, and `thinking.type: "disabled"` is no longer accepted. `output_config.effort` — one of `low`, `medium`, `high`, `xhigh`, `max` — is the control. **Omit it and you get `medium`**, so a request that sets no effort at all is not the cheapest or the deepest setting; name the level explicitly when either matters.

Effort scales reasoning adaptively rather than fixing a budget: on an easy prompt the model may spend very few thinking tokens even at high effort; on a hard one, higher effort produces more. The sweep below reports `usage.output_tokens_details.thinking_tokens` so you can see that directly.

In [ ]:
def invoke_effort(effort, prompt, model_id=CRIS_MODEL_ID, max_tokens=8000):
    """effort=None omits output_config entirely, which is how you observe the default."""
    body = {"anthropic_version": "bedrock-2023-05-31", "max_tokens": max_tokens,
            "messages": [{"role": "user", "content": prompt}]}
    if effort is not None:
        body["output_config"] = {"effort": effort}
    r = rt.invoke_model(modelId=model_id, contentType="application/json", accept="application/json",
                        body=json.dumps(body))
    res = json.loads(r["body"].read()); u = res.get("usage", {})
    return u.get("output_tokens"), (u.get("output_tokens_details") or {}).get("thinking_tokens")

reasoning_prompt = ("A Bedrock application serves 288,000 requests per day, spread uniformly across the day. "
    "Every request sends an identical 20,000-token system prompt, plus a 1,000-token user turn, "
    "and generates 500 output tokens.\n\n"
    "Pricing, per 1M tokens: uncached input $3.00, output $15.00, "
    "cache write $3.75, cache read $0.30. Prompt cache entries live for 5 minutes, "
    "so the system prompt is written once per 5-minute window and read by every other "
    "request in that window. Note that on Bedrock the reported input token count "
    "excludes cached tokens: cache reads and cache writes are billed as their own categories.\n\n"
    "1. Daily cost with no prompt caching.\n"
    "2. Daily cost with prompt caching enabled.\n"
    "3. The absolute and percentage savings.\n"
    "4. The number of requests per 5-minute window below which caching is more "
    "expensive than not caching, and why.\n\n"
    "Show your reasoning and state your assumptions.")

# The unset row should land on the medium row -- that is the default.
for e in [None, "low", "medium", "high", "xhigh", "max"]:
    out, think = invoke_effort(e, reasoning_prompt)
    label = "(unset -> default)" if e is None else e
    print(f"effort={label:18} output_tokens={out}  thinking_tokens={think}")


In [ ]:
# Confirm the removed controls: thinking cannot be turned off, and budgets are gone.
def try_body(label, extra):
    body = {"anthropic_version": "bedrock-2023-05-31", "max_tokens": 1024,
            "messages": [{"role": "user", "content": "Say ok"}], **extra}
    try:
        rt.invoke_model(modelId=CRIS_MODEL_ID, contentType="application/json",
                        accept="application/json", body=json.dumps(body))
        print(f"  {label:46} accepted")
    except Exception as e:
        print(f"  {label:46} rejected -- {str(e)[:100]}")

try_body("no thinking config (default: adaptive)", {})
try_body('thinking={"type":"disabled"}', {"thinking": {"type": "disabled"}})
try_body('thinking={"type":"enabled","budget_tokens":4096}',
         {"thinking": {"type": "enabled", "budget_tokens": 4096}})


### Sampling constraints

Sampling parameters not supported	temperature, top_p, or top_k set to any non-default value returns a 400 error. Remove them from all calls.

In [ ]:
def try_cfg(label, cfg, extra=None):
    kwargs = {"modelId": CRIS_MODEL_ID,
              "messages": [{"role": "user", "content": [{"text": "Say ok"}]}],
              "inferenceConfig": {"maxTokens": 1024, **cfg}}
    if extra:
        kwargs["additionalModelRequestFields"] = extra
    try:
        rt.converse(**kwargs)
        print(f"  {label:30} accepted")
    except Exception as e:
        print(f"  {label:30} rejected -- {str(e)[:95]}")

try_cfg("maxTokens only", {})
try_cfg("temperature=1.0", {"temperature": 1.0})
try_cfg("temperature=0.7", {"temperature": 0.7})
try_cfg("topP=0.99", {"topP": 0.99})
try_cfg("topP=0.5", {"topP": 0.5})
try_cfg("temperature + topP together", {"temperature": 1.0, "topP": 0.99})
try_cfg("topK=40", {}, {"top_k": 40})


## 11. Migrating from Claude Opus 5

1. **Model ID:** swap to `us.anthropic.claude-opus-5-5` or `global.anthropic.claude-opus-5-5` on `bedrock-runtime`, or `anthropic.claude-opus-5-5` on `bedrock-mantle`.
2. **Thinking config:** remove `thinking.type: "enabled"` / `"disabled"` and every manual `budget_tokens`. Thinking is always on and budgets are deprecated; `output_config.effort` is the only control. Any path that used to run with thinking disabled for cheap deterministic extraction now reasons — re-check its `max_tokens` headroom and latency.
3. **Effort calibration:** requests that set no effort run at the default, `medium`. Re-tune per workload: because Opus 5.5 spends tokens more efficiently, the level that was right on Opus 5 may now be higher than you need.
4. **Sampling:** strip `temperature`, `top_p`, and `top_k` unless the probe in section 3 shows they are accepted.
5. **Response parsing:** select the block whose type is `text`; never index a fixed position, since a reasoning block can come first.
6. **Refusals:** handle `stop_reason: "refusal"` (native) and `stopReason: "content_filtered"` (Converse) as normal paths, and re-run your prompt suite — cyber, bio, and AI-development prompts that passed on Opus 5 may now be blocked.
7. **Caching:** revisit where your cache points sit. Cheaper reads make a larger cached prefix worthwhile.
8. **Cost baselines:** recompute cost per task rather than reasoning from per-token price alone



## Summary

- Opus 5.5 reaches the same answers with fewer tokens, and lower prices plus much cheaper cache reads stack on top — measure cost per task, not price per token.
- Adaptive thinking is the only mode: no disable switch, no budgets, effort is the control.
- It reports back like a teammate — what it did, what it found, what it needs — which is what makes long-running sessions reviewable.
- Claude Fable 5.1-style classifiers in cyber security, biology, and AI development mean more refusals than any previous Opus; treat them as a response path and re-test before cutover.
- This notebook creates no AWS resources; it only invokes the model.

### Resources

- [Amazon Bedrock documentation](https://docs.aws.amazon.com/bedrock/latest/userguide/)
- [Anthropic model cards on Bedrock](https://docs.aws.amazon.com/bedrock/latest/userguide/model-cards-anthropic.html)
- [Amazon Bedrock pricing](https://aws.amazon.com/bedrock/pricing/)
